# DAF-06: Split by Time and Score the Baselines

Asha already has a simple forecasting habit: at 18:00 IST, she assumes
tomorrow's pollution will look like today's. Before we train a model, we need
to measure how well that free method works.

This notebook will use the completed daily table from DAF-05. We will inspect
how the data and Poor-or-worse days are distributed across months, choose one
chronological cut date, score two no-training baselines, and record their MAE
and Poor-day recall. We will also deliberately compare the honest time split
with a random split so we can see why shuffling a forecast dataset is unsafe.

## 1. Create the notebook and verify the Python kernel

Every later result depends on Python running in the selected notebook kernel.
We start with a tiny execution check before loading data or calculating any
metrics. If this cell does not run, the problem is the notebook environment,
not the forecasting logic.


In [3]:
print("Notebook ready")
print("Python kernel is executing code successfully.")


Notebook ready
Python kernel is executing code successfully.


## 2. Load the daily table and keep usable target rows

DAF-05 created one row per calendar day. Before scoring a forecast, we need
an actual value to compare against. That actual value is `target`, the next
day's PM2.5 mean.

### What we have before filtering

The saved DAF-05 file contains every calendar day in the full date span. It
has five columns:

| Column | Meaning | Keep for scoring? |
|---|---|---|
| `pm25_mean` | This day's full available PM2.5 mean | Yes, useful as a baseline input |
| `hours` | Number of hourly buckets with data | Yes, useful for checks and analysis |
| `pm25_until_17` | PM2.5 mean available by 17:00 IST | Yes, needed for Persistence |
| `valid` | Whether this day had at least 18 hours | Yes, useful for data-quality checks |
| `target` | Next day's PM2.5 mean | Yes, this is the answer we score against |

### What we remove from the scoring table

Some rows have an empty `target`. This happens when the next day is missing,
invalid, or there is no next day at the end of the dataset. We drop only those
rows from `daily_scoring`, because a prediction cannot be scored without an
actual answer.

We do **not** delete these rows from `daily_all` or from the raw archive. The
full DAF-05 table remains available for audit and later analysis. We are only
creating a smaller working table for fair metric calculation.

| Table | Contains | Purpose |
|---|---|---|
| `daily_all` | All 571 DAF-05 rows | Complete audit-friendly table |
| `daily_scoring` | Only rows with a usable `target` | Fair baseline scoring |

If we kept missing-target rows, MAE would compare predictions with missing
answers, and Poor-day recall would have an incorrect denominator. After this
step, every row in `daily_scoring` must have a real next-day target.

This step also checks that the input has the expected columns, unique dates,
and date range before we calculate any baseline.


In [4]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
input_path = project_root / "data/interim/daily_17.csv"
required_columns = {"pm25_mean", "hours", "pm25_until_17", "valid", "target"}

daily_all = pd.read_csv(input_path, parse_dates=["date"], index_col="date")

print("Input path:", input_path)
print("Full table shape:", daily_all.shape)
print("Full table columns:", daily_all.columns.tolist())
print("Full date range:", daily_all.index.min(), "to", daily_all.index.max())
print("Rows before target filtering:", len(daily_all))
print("Rows with missing target:", daily_all["target"].isna().sum())
print("Rows with usable target:", daily_all["target"].notna().sum())

assert required_columns.issubset(daily_all.columns), "DAF-05 column is missing"
assert daily_all.index.is_unique, "Duplicate dates found in DAF-05 output"

daily_scoring = daily_all.dropna(subset=["target"]).copy()

print("\nScoring table shape:", daily_scoring.shape)
print("Scoring date range:", daily_scoring.index.min(), "to", daily_scoring.index.max())
print("Scoring target missing values:", daily_scoring["target"].isna().sum())
print("\nScoring table preview:")
print(daily_scoring.head())


Input path: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/data/interim/daily_17.csv
Full table shape: (571, 5)
Full table columns: ['pm25_mean', 'hours', 'pm25_until_17', 'valid', 'target']
Full date range: 2025-02-19 00:00:00+05:30 to 2026-09-12 00:00:00+05:30
Rows before target filtering: 571
Rows with missing target: 84
Rows with usable target: 487

Scoring table shape: (487, 5)
Scoring date range: 2025-02-19 00:00:00+05:30 to 2026-09-10 00:00:00+05:30
Scoring target missing values: 0

Scoring table preview:
                           pm25_mean  hours  pm25_until_17  valid      target
date                                                                         
2025-02-19 00:00:00+05:30  78.782609     23      79.117647   True   49.416667
2025-02-20 00:00:00+05:30  49.416667     24      49.611111   True   90.375000
2025-02-21 00:00:00+05:30  90.375000     24      88.222222   True   69.125000
2025-02-22 00:00:00+05:30  69.125000     24      69.777778  

## 3. Inspect the calendar and Poor-or-worse days before splitting

We must choose the train/test cut from the calendar, not from whichever date
produces the most flattering baseline score. First we inspect how many usable
rows we have in each month and how many actual next-day targets are
Poor-or-worse.

For this project, Poor-or-worse means an actual `target >= 91 µg/m³`. This is
not a prediction yet: we are counting what really happened so we can choose a
test period that contains enough safety-relevant days for recall to mean
something.

The monthly table will contain:

| Value | Meaning |
|---|---|
| `rows` | Usable daily rows available in that month |
| `poor_days` | Actual next-day targets at or above 91 µg/m³ |
| `mean_target` | Average actual next-day PM2.5 for the month |

The chart shows the overall amount of data and the Poor-day count together.
We will use this calendar evidence to choose one chronological cut date in the
next step. We are not scoring baselines or trying multiple cut dates yet.


In [5]:
import plotly.graph_objects as go

poor_threshold = 91
monthly = daily_scoring.copy()
monthly["month"] = monthly.index.strftime("%Y-%m")
monthly["poor_or_worse"] = monthly["target"] >= poor_threshold

monthly_summary = (
    monthly.groupby("month")
    .agg(
        rows=("target", "size"),
        poor_days=("poor_or_worse", "sum"),
        mean_target=("target", "mean"),
    )
    .reset_index()
)

print("Poor-or-worse threshold:", poor_threshold, "µg/m³")
print("Number of months:", len(monthly_summary))
print("Total usable rows:", monthly_summary["rows"].sum())
print("Total Poor-or-worse days:", monthly_summary["poor_days"].sum())
print("\nMonthly summary:")
print(monthly_summary.to_string(index=False))

monthly_figure = go.Figure()
monthly_figure.add_trace(go.Bar(
    x=monthly_summary["month"],
    y=monthly_summary["rows"],
    name="Usable rows",
    customdata=monthly_summary[["poor_days", "mean_target"]],
    hovertemplate=(
        "Month: %{x}<br>"
        "Usable rows: %{y}<br>"
        "Poor-or-worse days: %{customdata[0]}<br>"
        "Mean target: %{customdata[1]:.2f} µg/m³<extra></extra>"
    ),
))
monthly_figure.add_trace(go.Scatter(
    x=monthly_summary["month"],
    y=monthly_summary["poor_days"],
    name="Poor-or-worse days",
    mode="lines+markers",
    yaxis="y2",
    hovertemplate="Month: %{x}<br>Poor-or-worse days: %{y}<extra></extra>",
))
monthly_figure.update_layout(
    title="Monthly Data Coverage and Poor-or-Worse Days",
    xaxis_title="Month",
    yaxis_title="Usable rows",
    yaxis2={
        "title": "Poor-or-worse days",
        "overlaying": "y",
        "side": "right",
    },
    hovermode="x unified",
)
monthly_figure.show()


Poor-or-worse threshold: 91 µg/m³
Number of months: 20
Total usable rows: 487
Total Poor-or-worse days: 156

Monthly summary:
  month  rows  poor_days  mean_target
2025-02    10          3    78.830072
2025-03    27          8    77.963231
2025-04    18          6    85.448594
2025-05    29          4    69.899182
2025-06    25          1    51.883670
2025-07    26          3    54.855547
2025-08    29          0    40.730684
2025-09    30          0    41.846457
2025-10    29         18   142.711948
2025-11    29         29   238.766349
2025-12    29         29   246.982553
2026-01    21         19   161.053694
2026-02    21         19   118.952295
2026-03    28          9    83.455576
2026-04    29          7    70.287226
2026-05    26          1    54.841723
2026-06    25          0    42.199661
2026-07    29          0    35.778335
2026-08    20          0    45.167393
2026-09     7          0    40.428860


## 4. Choose one chronological train/test cut

We now choose the split using the calendar evidence, before calculating any
baseline score. The rule is simple: older dates become training history, and
newer dates become the final test period.

### The cut we choose

We will use **2026-03-01** as the cut date:

- training: every usable row before 2026-03-01
- test: every usable row from 2026-03-01 onwards

This keeps the October 2025–February 2026 winter inside the training history,
so the future test period is not accidentally made entirely of summer days.
The test period also contains 17 actual Poor-or-worse days from March and
April 2026, so Poor-day recall has a real denominator.

We must not shuffle these rows. In a real forecast, Asha predicts a future day
using information available in the past. A random split could put a later day
in training and an earlier day in test, allowing the experiment to learn from
the future.

The code will print the size, date range, and Poor-day count on each side, then
show the split on a timeline. The test must begin after the training period
ends, and both sets must remain separate.


In [6]:
cut_date = pd.Timestamp("2026-03-01", tz=daily_scoring.index.tz)

daily_scoring["poor_or_worse"] = daily_scoring["target"] >= poor_threshold
train = daily_scoring[daily_scoring.index < cut_date].copy()
test = daily_scoring[daily_scoring.index >= cut_date].copy()

print("Chosen cut date:", cut_date)
print("\nTrain rows:", len(train))
print("Train date range:", train.index.min(), "to", train.index.max())
print("Train Poor-or-worse days:", train["poor_or_worse"].sum())
print("\nTest rows:", len(test))
print("Test date range:", test.index.min(), "to", test.index.max())
print("Test Poor-or-worse days:", test["poor_or_worse"].sum())

assert len(train) > 0 and len(test) > 0, "Train or test set is empty"
assert train.index.max() < test.index.min(), "Train and test dates overlap"
assert test["poor_or_worse"].sum() > 0, "Test has no Poor-or-worse days"

split_figure = go.Figure()
split_figure.add_trace(go.Scatter(
    x=train.index,
    y=train["target"],
    mode="markers",
    name="Train rows",
    customdata=train[["poor_or_worse"]],
    hovertemplate=(
        "Date: %{x|%Y-%m-%d}<br>"
        "Target: %{y:.2f} µg/m³<br>"
        "Poor-or-worse: %{customdata[0]}<extra></extra>"
    ),
))
split_figure.add_trace(go.Scatter(
    x=test.index,
    y=test["target"],
    mode="markers",
    name="Test rows",
    customdata=test[["poor_or_worse"]],
    hovertemplate=(
        "Date: %{x|%Y-%m-%d}<br>"
        "Target: %{y:.2f} µg/m³<br>"
        "Poor-or-worse: %{customdata[0]}<extra></extra>"
    ),
))
split_figure.add_vline(
    x=cut_date,
    line_dash="dash",
    annotation_text="Cut date: 2026-03-01",
)
split_figure.update_layout(
    title="Chronological Train/Test Split",
    xaxis_title="Date",
    yaxis_title="Actual next-day PM2.5 target (µg/m³)",
    hovermode="closest",
)
split_figure.show()


Chosen cut date: 2026-03-01 00:00:00+05:30

Train rows: 323
Train date range: 2025-02-19 00:00:00+05:30 to 2026-02-28 00:00:00+05:30
Train Poor-or-worse days: 139

Test rows: 164
Test date range: 2026-03-01 00:00:00+05:30 to 2026-09-10 00:00:00+05:30
Test Poor-or-worse days: 17


## 5. Score the two no-training baselines

### The one question this step answers

It is 18:00 on Mar 1. Asha has to say how bad tomorrow's air (Mar 2) will be.
She has not trained any model. She just guesses with a simple rule.

**Step 5 asks: how wrong is that simple guess, on average?**

That number becomes the score to beat. If a trained model later cannot beat a
free guess, the model is useless.

### Picture it on a timeline

```text
        Feb 28            Mar 1                         Mar 2
   ┌──────────────┐ ┌──────────────────────┐    ┌──────────────────┐
   │ full day done│ │ 00:00 ... 17:00 │18:00│    │ the real answer  │
   │ pm25_mean    │ │  pm25_until_17  │ ◄── Asha │ = target on Mar 1│
   └──────┬───────┘ └────────┬────────┴─────┘ │  └────────▲─────────┘
          │                  │          predicts          │
          │  Guess B         │  Guess A      ─────────────┘
          └── "yesterday     └── "persistence"    compare guess vs real
               full day"
```

Asha stands at **18:00 on Mar 1**. She can only look **backwards**:

| Guess | What Asha looks at | Rule in plain English |
|---|---|---|
| **A. Persistence** | Mar 1, the hours until 17:00 (`pm25_until_17`) | "Tomorrow will be like today so far." |
| **B. Yesterday full day** | Feb 28, the whole finished day (`pm25_mean` of D − 1) | "Tomorrow will be like yesterday." |

Why not use Mar 1's full-day `pm25_mean`? Because at 18:00 the day is not
over yet. Using it would be cheating with information from the future.

### A small worked example

Say this is the real data (made-up numbers):

| date | `pm25_mean` (full day) | `pm25_until_17` | `target` (= next day's `pm25_mean`) |
|---|---:|---:|---:|
| Feb 28 | 100 | — | 95 |
| Mar 1 | 95 | 85 | **120** |
| Mar 2 | 120 | 110 | **98** |
| Mar 3 | 98 | 92 | **60** |
| Mar 4 | 60 | — | — |

Notice that `target` on Mar 1 is just Mar 2's `pm25_mean`. DAF-05 already
put tomorrow's answer on today's row, so we never move `target` again.

Now Step 5 builds a new table called `predictions`. One row = one forecast:

| forecast made on | `actual` (copy of `target`) | `persistence` (copy of `pm25_until_17`) | `yesterday_full_day` (`pm25_mean` shifted down 1 row) |
|---|---:|---:|---:|
| Mar 1 | 120 | 85 | 100 (from Feb 28) |
| Mar 2 | 98 | 110 | 95 (from Mar 1) |
| Mar 3 | 60 | 92 | 120 (from Mar 2) |

In code:

```python
predictions["actual"]             = test["target"]                  # the real answer
predictions["persistence"]        = test["pm25_until_17"]           # same row, no shift
predictions["yesterday_full_day"] = daily_all["pm25_mean"].shift(1) # previous row
```

We shift on the **full** `daily_all` table first and only then keep the test
dates. Otherwise the first test day (Mar 1) would have no "yesterday",
because Feb 28 lives in the training part.

Follow the arrows. Each colour is one column, and it shows where that value comes from:

![Where each value in predictions comes from](figures/daf06_step5_value_flow.png)

In one line each:

- **Green, `actual`:** stays on the same row. Mar 1's `target` (120) is copied as it is.
- **Blue, `persistence`:** stays on the same row. Mar 1's `pm25_until_17` (85) is copied as it is.
- **Orange, `yesterday_full_day`:** comes from **one row up**. Mar 1 takes Feb 28's `pm25_mean` (100).
  That "one row up" is exactly what `.shift(1)` does.
- **Red dashed line:** the train/test cut. Feb 28 is on the train side. If we
  cut first and shifted afterwards, Mar 1 would have no row above it and would get `NaN`.
  That is why we shift `daily_all` first and pick the test dates after.

### Score 1: MAE, "how far off, on average?"

Take each guess, subtract the real answer, drop the minus sign, and average.

The whole idea on one page. Read the table first, then the gaps on the right, then the four steps:

![MAE worked out by hand](figures/daf06_step5_mae.png)

First, the names used in the formula:

- $y_i$ is the **real answer** for row $i$. It is the `actual` column.
- $\hat{y}_i$ ("y-hat") is the **guess** for row $i$. It is the `persistence` or `yesterday_full_day` column.
- The hat means "estimated". No hat means the real value.

Here is the same table with those names written on it. We score persistence first:

| $i$ | forecast made on | $y_i$ = `actual` | $\hat{y}_i$ = `persistence` | $\lvert\hat{y}_i - y_i\rvert$ = error |
|---:|---|---:|---:|---:|
| 1 | Mar 1 | $y_1$ = 120 | $\hat{y}_1$ = 85 | \|85 − 120\| = **35** |
| 2 | Mar 2 | $y_2$ = 98 | $\hat{y}_2$ = 110 | \|110 − 98\| = **12** |
| 3 | Mar 3 | $y_3$ = 60 | $\hat{y}_3$ = 92 | \|92 − 60\| = **32** |

Now plug the numbers into the formula. We have $n = 3$ rows:

$$
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|\hat{y}_i-y_i|
= \frac{1}{3}\big(\,|\hat{y}_1-y_1| + |\hat{y}_2-y_2| + |\hat{y}_3-y_3|\,\big)
= \frac{35 + 12 + 32}{3} = \frac{79}{3} \approx \mathbf{26.3}
$$

Then do the same for `yesterday_full_day`. Only the $\hat{y}$ column changes. The real answers $y_i$ stay the same:

| $i$ | forecast made on | $y_i$ = `actual` | $\hat{y}_i$ = `yesterday_full_day` | $\lvert\hat{y}_i - y_i\rvert$ = error |
|---:|---|---:|---:|---:|
| 1 | Mar 1 | $y_1$ = 120 | $\hat{y}_1$ = 100 | \|100 − 120\| = **20** |
| 2 | Mar 2 | $y_2$ = 98 | $\hat{y}_2$ = 95 | \|95 − 98\| = **3** |
| 3 | Mar 3 | $y_3$ = 60 | $\hat{y}_3$ = 120 | \|120 − 60\| = **60** |

$$
\text{MAE} = \frac{20 + 3 + 60}{3} = \frac{83}{3} \approx \mathbf{27.7}
$$

In code, that is one line per baseline:

```python
mean_absolute_error(scored["actual"], scored["persistence"])   # (y, y_hat)
```

An MAE of 26 means "on a normal day, this guess is about 26 µg/m³ wrong".
**Lower is better.** Why drop the minus sign? A guess that is 35 too low and one
that is 35 too high are equally wrong. Without the absolute value they would
cancel out, and the error would look like 0.

### Score 2: Poor-day recall, "did we warn people on the bad days?"

#### Why do we need a second score at all?

Think about who reads Asha's 18:00 forecast: parents of children with asthma.
They decide tonight whether the child plays outside tomorrow.

Asha can make two kinds of mistake, and they do not cost the same:

| Mistake | What happens | Cost |
|---|---|---|
| **Missed Poor day**: the air was bad, but no warning was given | The child plays outside in bad air | 🔴 Very costly, a health risk |
| **False alarm**: a warning was given, but the air was clean | The child stays in for nothing | 🟠 Annoying, but safe |

MAE cannot see this difference. For MAE, being 35 too low on a Poor day is the
same as being 35 too high on a clean day. Both are just "35".

So we add a second score that looks **only at the dangerous days** and asks one
question: **out of all the real Poor days, how many did we warn about?** That
score is recall.

#### How it is counted

A day is Poor-or-worse when PM2.5 ≥ 91. We use the same line for both sides:

- the **real answer** is Poor if `actual` ≥ 91;
- the **guess** is a warning if the guess ≥ 91.

Each row then falls into one of four boxes:

| | Warned (guess ≥ 91) | Not warned (guess < 91) |
|---|---|---|
| **Really Poor** (actual ≥ 91) | ✅ CAUGHT | ❌ MISSED |
| **Really clean** (actual < 91) | ⚠️ false alarm | quiet, ok |

Recall uses **only the top row**, the real Poor days:

$$
\text{Recall} = \frac{\text{CAUGHT}}{\text{CAUGHT} + \text{MISSED}}
= \frac{\text{Poor days we warned about}}{\text{all real Poor days}}
$$

#### The full picture, step by step

![Poor-day recall worked out by hand](figures/daf06_step5_recall.png)

Reading the table row by row:

| date | $y$ actual | Really Poor? | persistence $\hat{y}$ | result | yesterday $\hat{y}$ | result |
|---|---:|---|---:|---|---:|---|
| Mar 1 | 120 | ✅ yes | 85 (< 91) | ❌ **MISSED** | 100 (≥ 91) | ✅ CAUGHT |
| Mar 2 | 98 | ✅ yes | 110 (≥ 91) | ✅ CAUGHT | 95 (≥ 91) | ✅ CAUGHT |
| Mar 3 | 60 | no | 92 (≥ 91) | ⚠️ false alarm | 120 (≥ 91) | ⚠️ false alarm |

- **Persistence:** 1 caught out of 2 Poor days, so 1 / (1 + 1) = **50 %**. It missed Mar 1.
- **Yesterday full day:** 2 caught out of 2 Poor days, so 2 / (2 + 0) = **100 %**.
- **Mar 3 does not count** for recall, because it was not a real Poor day. Both
  false alarms there are simply ignored. **Higher recall is better.**

#### The two scores disagree, and that is the point

| Score | persistence | yesterday_full_day | Winner |
|---|---:|---:|---|
| MAE (lower is better) | **26.3** | 27.7 | persistence: closer on average |
| Recall (higher is better) | 50 % | **100 %** | yesterday: never missed a bad day |

If we looked only at MAE, we would pick persistence and never notice that it
missed a dangerous day. That is why we measure recall.

**Trap:** a forecast that *always* says "Poor!" gets 100 % recall, because it
can never miss. But it cries wolf on every clean day. So recall is never read
alone. MAE (and, later, counting false alarms) keeps it honest.

In the code, the four boxes come from two True/False columns:

```python
actual_poor    = scored["actual"]   >= poor_threshold   # really Poor?
predicted_poor = scored[model_name] >= poor_threshold   # did we warn?
correctly_flagged = (actual_poor & predicted_poor).sum()  # CAUGHT
recall = correctly_flagged / actual_poor.sum()           # CAUGHT / all real Poor
```

### What to take away

```text
simple guess ──► compare with the real next day ──► two numbers
                                                     MAE    (how far off)
                                                     Recall (did we catch bad days)
```

These two numbers are the **bar**. Every model in the next tickets must beat
them on the same test days, or it is not worth running.

The code below does exactly this on the real test period. It skips rows where a
guess or the answer is missing, prints both scores, and draws a chart. Hover
over a date to see the actual value next to both guesses.

In [7]:
from sklearn.metrics import mean_absolute_error

baseline_names = ["persistence", "yesterday_full_day"]

# ── Step 5.1  Build the guesses ─────────────────────────────────────────────
print(daily_all)

# Read the next line left to right, in 3 small moves:
#
#   daily_all["pm25_mean"]     -> one column, ALL dates (train + test), one row per day
#
#       date     pm25_mean
#       Feb 27      110
#       Feb 28      100
#       Mar 1        95
#       Mar 2       120
#
#   .shift(1)                  -> every value slides DOWN by one row.
#                                 The dates stay where they are; only the numbers move.
#
#       date     pm25_mean      after .shift(1)
#       Feb 27      110    ->     NaN   (nothing above the first row)
#       Feb 28      100    ->     110   (Feb 27's value)
#       Mar 1        95    ->     100   (Feb 28's value)  = "yesterday" for Mar 1
#       Mar 2       120    ->      95   (Mar 1's value)   = "yesterday" for Mar 2
#
#   .reindex(test.index)       -> keep ONLY the test dates (Mar 1 onwards), matched by date.
#
#       date     yesterday_full_day
#       Mar 1        100
#       Mar 2         95
#
# Why shift BEFORE reindex?  Feb 28 is in the train part. If we kept only the
# test dates first, Mar 1 would have no row above it and would get NaN.
#
# Careful: shift(1) means "one ROW up", not "one DAY back". Here they are the
# same only because DAF-05 kept one row for EVERY calendar day (no gaps).
yesterday_full_day = daily_all["pm25_mean"].shift(1).reindex(test.index)
print("------------------")
print(yesterday_full_day)

print("------------------")
# ── Step 5.2  Put y and both ŷ on the same row ──────────────────────────────
predictions = pd.DataFrame(index=test.index)
print(predictions)
predictions["actual"] = test["target"]
predictions["persistence"] = test["pm25_until_17"] 
predictions["yesterday_full_day"] = yesterday_full_day  

print(predictions)



                           pm25_mean  hours  pm25_until_17  valid      target
date                                                                         
2025-02-19 00:00:00+05:30  78.782609     23      79.117647   True   49.416667
2025-02-20 00:00:00+05:30  49.416667     24      49.611111   True   90.375000
2025-02-21 00:00:00+05:30  90.375000     24      88.222222   True   69.125000
2025-02-22 00:00:00+05:30  69.125000     24      69.777778   True   74.000000
2025-02-23 00:00:00+05:30  74.000000     24      70.277778   True  101.166667
...                              ...    ...            ...    ...         ...
2026-09-08 00:00:00+05:30  48.333333     12      45.444444  False         NaN
2026-09-09 00:00:00+05:30  48.933333     15      35.777778  False   38.333333
2026-09-10 00:00:00+05:30  38.333333     24      35.111111   True   42.863636
2026-09-11 00:00:00+05:30  42.863636     22      50.687500   True         NaN
2026-09-12 00:00:00+05:30  13.000000      1      13.000000  Fals

In [31]:
# ── Step 5.3  Look at the rows we cannot score, before dropping them ────────
missing_rows = predictions[predictions.isna().any(axis=1)]
print("Test rows available:", len(predictions))
print("Rows with a missing value:", len(missing_rows))
print(missing_rows.to_string())

# ── Step 5.4  Score both baselines on the SAME rows ─────────────────────────
scored = predictions.dropna().copy()
print("Acrtual Scored",scored)
actual_poor = scored["actual"] >= poor_threshold
print("\nRows scored (same for both baselines):", len(scored))
print("Actual Poor-or-worse days in scored rows:", actual_poor.sum())   



# ── Step 5.5  MAE + the four boxes for each baseline ────────────────────────
baseline_results = []
for model_name in baseline_names:
    predicted_poor = scored[model_name] >= poor_threshold  # did we warn?

    caught = (actual_poor & predicted_poor).sum()           # ✅ Poor and warned
    missed = (actual_poor & ~predicted_poor).sum()          # ❌ Poor, no warning
    false_alarms = (~actual_poor & predicted_poor).sum()    # ⚠️ clean, but warned

    baseline_results.append({
        "model": model_name,
        "rows_scored": len(scored),
        "mae": mean_absolute_error(scored["actual"], scored[model_name]),  # (y, ŷ)
        "poor_days": caught + missed,
        "caught": caught,
        "missed": missed,
        "false_alarms": false_alarms,
        "recall_poor": caught / (caught + missed) if caught + missed else float("nan"),
    })

baseline_results = pd.DataFrame(baseline_results)

print("\nBaseline results:")
print(baseline_results.to_string(index=False, float_format="%.3f"))

assert len(scored) > 0, "No scorable rows"
assert (baseline_results["poor_days"] > 0).all(), "No Poor-day denominator"
assert (baseline_results["caught"] + baseline_results["missed"]
        == actual_poor.sum()).all(), "Boxes do not add up to the Poor days"

Test rows available: 164
Rows with a missing value: 3
                              actual  persistence  yesterday_full_day
date                                                                 
2026-06-21 00:00:00+05:30  29.130435          NaN            48.73913
2026-06-25 00:00:00+05:30  30.166667          NaN                 NaN
2026-08-09 00:00:00+05:30  43.340909        26.35                 NaN
Acrtual Scored                               actual  persistence  yesterday_full_day
date                                                                 
2026-03-01 00:00:00+05:30  99.950000    83.647059          111.260870
2026-03-02 00:00:00+05:30  95.458333    93.750000           87.347826
2026-03-03 00:00:00+05:30  76.666667   100.666667           99.950000
2026-03-04 00:00:00+05:30  68.791667    78.944444           95.458333
2026-03-05 00:00:00+05:30  87.500000    67.888889           76.666667
...                              ...          ...                 ...
2026-09-03 00:00:00+0

In [9]:
# ── Step 5.6  Chart: actual vs guesses, with the MISSED Poor days marked ────
baseline_figure = go.Figure()
baseline_figure.add_trace(go.Scatter(
    x=predictions.index,
    y=predictions["actual"],
    mode="lines+markers",
    name="Actual target",
    hovertemplate="Date: %{x|%Y-%m-%d}<br>Actual: %{y:.2f} µg/m³<extra></extra>",
))
for model_name in baseline_names:
    baseline_figure.add_trace(go.Scatter(
        x=predictions.index,
        y=predictions[model_name],
        mode="lines+markers",
        name=model_name,
        hovertemplate="Date: %{x|%Y-%m-%d}<br>" + model_name + ": %{y:.2f} µg/m³<extra></extra>",
    ))

    missed_days = scored[actual_poor & (scored[model_name] < poor_threshold)]
    baseline_figure.add_trace(go.Scatter(
        x=missed_days.index,
        y=missed_days["actual"],
        mode="markers",
        marker=dict(symbol="x", size=12),
        name=f"Missed by {model_name} ({len(missed_days)})",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Missed Poor day: %{y:.2f} µg/m³<extra></extra>",
    ))

baseline_figure.add_hline(
    y=poor_threshold,
    line_dash="dash",
    annotation_text="Poor threshold: 91 µg/m³",
)
baseline_figure.update_layout(
    title="Test Period: Actual Targets versus Simple Baselines",
    xaxis_title="Date",
    yaxis_title="PM2.5 (µg/m³)",
    hovermode="x unified",
)
baseline_figure.show()

## Where we are, and what is left

### What we have done so far (Steps 1–5)

| Step | What we did | What we learned |
|---|---|---|
| 1 | Checked the Python kernel | The notebook runs |
| 2 | Loaded `daily_17.csv` and kept only rows with a real `target` | 571 calendar days, of which the ones with a usable next day form `daily_scoring` |
| 3 | Counted rows and Poor-or-worse days per month | The data holds one winter, so the cut date must leave Poor days in the test period |
| 4 | Cut by time at **2026-03-01** | Train = before the cut, test = from the cut onwards, no overlap |
| 5 | Scored two free guesses on the test rows | Persistence MAE ≈ 15.73, yesterday's full day MAE ≈ 16.40 (161 scored rows) |

So Asha's free habit is already about **16 µg/m³ off on a typical day**. That is the bar every model must beat.

### What is left (from the DAF-06 ticket)

| Step | What we will do | Why |
|---|---|---|
| **6** | Save both baseline scores to `reports/experiments.csv` | Every later ticket adds a row to this file, so we can always compare a new model with the baselines |
| **7** | Random 80/20 split + `LinearRegression` on `pm25_until_17`, compared with the time split | Shows with our own numbers why shuffling a forecast dataset is unsafe |
| **8** | Write the *My notes* answers: cut date and why, sizes, Poor days, both MAEs and recalls, what the random split showed | Acceptance criteria and explain-back for the ticket |

We do one step at a time, and each step prints its details so we can check it before moving on.

## 6. Start the experiment log

### The one question this step answers

Asha has two scores now. In two weeks she will have ten models and will have
forgotten which score belonged to which. **Where do we write the results down so
they are never lost?**

### The idea

An experiment log is one CSV file where every run is one row. Think of it as an
audit table in a database: you never overwrite history, you only add rows.

```text
baseline_results (in memory, lost when the kernel stops)
        │  pick the columns the log needs
        ▼
reports/experiments.csv   (on disk, kept in git)
```

The columns are fixed by the ticket:

| Column | Meaning |
|---|---|
| `ticket` | Which ticket produced this row (`DAF-06`) |
| `date` | The day the run was made |
| `model` | `persistence` or `yesterday_full_day` |
| `features` | Which input columns the guess used |
| `mae` | Mean absolute error on the test rows (lower is better) |
| `recall_poor` | Share of real Poor days we warned about (higher is better) |
| `notes` | The cut date, rows scored and Poor-day count, so the row explains itself |

### Two safety rules in the code

1. **Re-running must not create duplicates.** If the notebook is run twice, we
   first remove old `DAF-06` rows for these two models, then add fresh ones.
   Other tickets' rows are never touched.
2. **Check before trusting.** We assert that exactly two `DAF-06` rows exist,
   and print the file back from disk so we see what was really saved.

In [10]:
# ── Step 6.1  Decide where the log lives ────────────────────────────────────
reports_dir = project_root / "reports"
reports_dir.mkdir(exist_ok=True)
log_path = reports_dir / "experiments.csv"
log_columns = ["ticket", "date", "model", "features", "mae", "recall_poor", "notes"]

features_used = {
    "persistence": "pm25_until_17 (same day, hours 00-17)",
    "yesterday_full_day": "pm25_mean of previous day",
}
print("Log file:", log_path)
print("Log already exists:", log_path.exists())

# ── Step 6.2  Turn each baseline result into one log row ────────────────────
run_date = pd.Timestamp.today().strftime("%Y-%m-%d")
new_rows = []
for _, result in baseline_results.iterrows():
    new_rows.append({
        "ticket": "DAF-06",
        "date": run_date,
        "model": result["model"],
        "features": features_used[result["model"]],
        "mae": round(result["mae"], 3),
        "recall_poor": round(result["recall_poor"], 3),
        "notes": (
            f"time split, cut {cut_date.date()}; "
            f"{int(result['rows_scored'])} test rows; "
            f"{int(result['poor_days'])} Poor days "
            f"(caught {int(result['caught'])}, missed {int(result['missed'])}, "
            f"false alarms {int(result['false_alarms'])})"
        ),
    })
new_rows = pd.DataFrame(new_rows, columns=log_columns)
print("\nRows we are about to log:")
print(new_rows.to_string(index=False))

# ── Step 6.3  Add them without duplicating on a re-run ──────────────────────
if log_path.exists():
    old_log = pd.read_csv(log_path)
    is_rerun = (old_log["ticket"] == "DAF-06") & old_log["model"].isin(baseline_names)
    print("\nOld log rows:", len(old_log), "| DAF-06 baseline rows replaced:", is_rerun.sum())
    log = pd.concat([old_log[~is_rerun], new_rows], ignore_index=True)
else:
    print("\nNo old log, creating it.")
    log = new_rows

log.to_csv(log_path, index=False)

# ── Step 6.4  Read it back from disk and check ──────────────────────────────
saved_log = pd.read_csv(log_path)
print("\nexperiments.csv as saved on disk:")
print(saved_log.to_string(index=False))

assert list(saved_log.columns) == log_columns, "Log columns differ from the ticket"
assert (saved_log["ticket"] == "DAF-06").sum() == 2, "Expected exactly two DAF-06 rows"
assert not saved_log.duplicated(["ticket", "model"]).any(), "Duplicate ticket/model rows"
print("\nLog check passed: 2 baseline rows, no duplicates.")

# ── Step 6.5  Picture: both scores side by side ─────────────────────────────
log_figure = go.Figure()
log_figure.add_trace(go.Bar(
    x=saved_log["model"], y=saved_log["mae"], name="MAE (lower is better)",
    hovertemplate="%{x}<br>MAE: %{y:.2f} µg/m³<extra></extra>",
))
log_figure.add_trace(go.Bar(
    x=saved_log["model"], y=saved_log["recall_poor"] * 100,
    name="Poor-day recall % (higher is better)",
    hovertemplate="%{x}<br>Recall: %{y:.1f}%<extra></extra>",
))
log_figure.update_layout(
    title="The bar to beat: baseline scores saved in experiments.csv",
    xaxis_title="Baseline", yaxis_title="MAE in µg/m³  |  recall in %",
    barmode="group",
)
log_figure.show()

Log file: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/reports/experiments.csv
Log already exists: True

Rows we are about to log:
ticket       date              model                              features    mae  recall_poor                                                                                         notes
DAF-06 2026-09-20        persistence pm25_until_17 (same day, hours 00-17) 15.728        0.471 time split, cut 2026-03-01; 161 test rows; 17 Poor days (caught 8, missed 9, false alarms 12)
DAF-06 2026-09-20 yesterday_full_day             pm25_mean of previous day 16.399        0.471 time split, cut 2026-03-01; 161 test rows; 17 Poor days (caught 8, missed 9, false alarms 10)

Old log rows: 2 | DAF-06 baseline rows replaced: 2

experiments.csv as saved on disk:
ticket       date              model                              features    mae  recall_poor                                                                                       

## 7. Train our first model

### The question

Asha's friend Ravi says: *"Why not let a model learn the rule, instead of using
a fixed guess?"* Step 7 trains the simplest model we have and asks: **does
learning from data beat Asha's free rule?**

### What "training" means

Persistence is a rule someone wrote by hand: `prediction = today's value`.
A model finds the rule itself from past rows. We use `LinearRegression` with
one input:

```text
input  X = pm25_until_17   (today's air until 17:00)
answer y = target          (tomorrow's PM2.5)

the model learns a straight line:  prediction = slope × X + intercept
```

Persistence is the same line with the numbers fixed at slope 1 and intercept 0.
The model is allowed to choose better numbers from the training rows.

In Java terms: `fit(X, y)` learns the two numbers, `predict(X)` applies them.

### The plan, one small step at a time

| Step | What we do |
|---|---|
| 7.1 | Pick the input `X` and the answer `y` |
| 7.2 | Split by **time**, using the same `cut_date` from Step 4 |
| 7.3 | `fit` on the train rows |
| 7.4 | `predict` on the test rows and score with MAE |
| 7.5 | Compare with persistence on the same rows |

### The rule

The model learns only from train rows. We score only on test rows it has never seen.

### 7.1 Pick the input `X` and the answer `y`

The model needs two things:

- `X`, the input it looks at. Here it is one column, `pm25_until_17`.
- `y`, the answer it must predict. Here it is `target`.

`X` uses double brackets `[[ ]]` because scikit-learn expects a **table**, even
with one column. `y` uses single brackets because it is one column.

We drop the rows where `pm25_until_17` is missing, because the model cannot
learn from or predict on an empty input. We print a few rows to check that
`X` and `y` line up on the same dates.

In [15]:
# Keep only the days where today's 00-17 average exists, the model cannot use an empty input
model_rows = daily_scoring.dropna(subset=["pm25_until_17"])

X = model_rows[["pm25_until_17"]]
y = model_rows["target"]

print(X.head())
print(y.head())
print("Rows:", len(model_rows))


                           pm25_until_17
date                                    
2025-02-19 00:00:00+05:30      79.117647
2025-02-20 00:00:00+05:30      49.611111
2025-02-21 00:00:00+05:30      88.222222
2025-02-22 00:00:00+05:30      69.777778
2025-02-23 00:00:00+05:30      70.277778
date
2025-02-19 00:00:00+05:30     49.416667
2025-02-20 00:00:00+05:30     90.375000
2025-02-21 00:00:00+05:30     69.125000
2025-02-22 00:00:00+05:30     74.000000
2025-02-23 00:00:00+05:30    101.166667
Name: target, dtype: float64
Rows: 482


### 7.2 Split by time

The model must learn from the past and be tested on the future, so we reuse the
same `cut_date` from Step 4 (2026-03-01):

```text
X, y  ──►  before 2026-03-01  ──►  X_train, y_train   (the model learns from these)
      └─►  from 2026-03-01    ──►  X_test,  y_test    (the model never sees these while learning)
```

We split `X` and `y` with the same date rule, so each input stays matched with
its own answer. The cut date is reused, not chosen again, so we cannot tune it
to flatter the score.

We print the size and date range of each side. Train must end before test begins.

In [20]:
# Older rows (before the cut date) are for learning: the model is allowed to see these
X_train = X[X.index < cut_date]
y_train = y[y.index < cut_date]

# Newer rows (from the cut date onwards) are for scoring: the model must not see these while learning
X_test = X[X.index >= cut_date]
y_test = y[y.index >= cut_date]

print("Train rows:", len(X_train), "| from", X_train.index.min().date(), "to", X_train.index.max().date())
print("Test rows: ", len(X_test), "| from", X_test.index.min().date(), "to", X_test.index.max().date())

# Safety check: the last train day must be before the first test day
assert X_train.index.max() < X_test.index.min(), "Train and test overlap"



Train rows: 320 | from 2025-02-19 to 2026-02-28
Test rows:  162 | from 2026-03-01 to 2026-09-10


### 7.3 Train the model with `fit`

Now the model learns. `LinearRegression` looks at the train rows and finds the
straight line that fits them best:

```text
prediction = slope × pm25_until_17 + intercept
```

Two lines of code do it:

```python
model = LinearRegression()      # create an empty model, it knows nothing yet
model.fit(X_train, y_train)     # learn slope and intercept from the train rows
```

After `fit`, the model holds two learned numbers:

| Number | Meaning | Persistence uses |
|---|---|---|
| `slope` (`coef_`) | how much tomorrow rises for each 1 µg/m³ today | 1 |
| `intercept` (`intercept_`) | the starting level when today is 0 | 0 |

If the learned slope is close to 1 and the intercept close to 0, the model has
just rediscovered persistence. Any difference is what the model learned from the
data. We print both numbers and draw the learned line over the train rows.

Only `X_train` and `y_train` are used here. The test rows are not touched.

In [24]:
from sklearn.linear_model import LinearRegression

# Create an empty model, it knows nothing yet
model = LinearRegression()

# Learn the best straight line from the train rows only
model.fit(X_train, y_train)

# The two numbers the model learned
slope = model.coef_[0]
intercept = model.intercept_

print("Learned line: prediction =", round(slope, 3), "× pm25_until_17 +", round(intercept, 2))
print("Persistence would be:      prediction = 1 × pm25_until_17 + 0")

Learned line: prediction = 0.797 × pm25_until_17 + 27.08
Persistence would be:      prediction = 1 × pm25_until_17 + 0


#### See the line, and why the model chose these numbers

Every dot is one day in the train rows: today's air on the x-axis, tomorrow's
real air on the y-axis. A line is a guess for every day. The **gap** between a
dot and the line is that day's error.

```text
tomorrow                      ●  ← real value
   │                          │  gap = how wrong the line was
   │              ─────────── ●──── line's prediction
   │
   └────────────────────────────  today
```

We could draw thousands of different lines, one for every pair of slope and
intercept. The model does not pick at random. It picks the pair that makes the
gaps **as small as possible overall**. `LinearRegression` does this by
minimising the sum of the **squared** gaps ("least squares").

Why squared? A gap of 10 becomes 100 and a gap of 30 becomes 900, so one big
miss counts far more than several small ones. The line is forced to stay close
to every dot, not only most of them.

The cell below tries four lines and prints how far each one is from the dots:

| Line | Slope | Intercept |
|---|---:|---:|
| Persistence | 1.00 | 0 |
| Learned by the model | 0.80 | 27.1 |
| A guess | 0.50 | 20 |
| A guess | 1.20 | 0 |

The learned line has the smallest gap. No other pair of numbers can beat it on
these train rows. That is why the model ended up with these two numbers.

The model minimises squared gaps on the **train** rows, not MAE on the test
rows. It is a good fit for the past, and Step 7.4 will show how well it holds up
on the future.

In [27]:
# One real day to explain the graph: the day whose "today" value is closest to 100
example_day = (X_train["pm25_until_17"] - 100).abs().idxmin()
today_value = X_train.loc[example_day, "pm25_until_17"]
real_value = y_train.loc[example_day]

# The two guesses for tomorrow
persistence_guess = today_value                    # Asha's rule: tomorrow = today
learned_guess = slope * today_value + intercept    # the model: slope × today + intercept

print("Example day:", example_day.date())
print(f"Today until 17:00 = {today_value:.0f}   |   real tomorrow = {real_value:.0f}")
print(f"Persistence says {persistence_guess:.0f}  -> off by {abs(persistence_guess - real_value):.0f}")
print(f"Model says {learned_guess:.0f}  -> off by {abs(learned_guess - real_value):.0f}")

gap_figure = go.Figure()

# Grey dots: every train day (across = today, up = tomorrow's real value)
gap_figure.add_trace(go.Scatter(
    x=X_train["pm25_until_17"], y=y_train, mode="markers", name="Train days (1 dot = 1 day)",
    marker=dict(color="lightgrey", size=7),
    hovertemplate="Today: %{x:.0f}<br>Tomorrow: %{y:.0f}<extra></extra>",
))

# The two lines, with their formulas in the legend
x_line = [40, 160]
gap_figure.add_trace(go.Scatter(
    x=x_line, y=x_line, mode="lines", line=dict(color="royalblue", dash="dash"),
    name="Persistence: 1.00 × today + 0",
))
gap_figure.add_trace(go.Scatter(
    x=x_line, y=[slope * v + intercept for v in x_line], mode="lines", line=dict(color="darkorange"),
    name=f"Learned: {slope:.2f} × today + {intercept:.1f}",
))

# Red dotted line = the gap between the real value and the model's guess
gap_figure.add_trace(go.Scatter(
    x=[today_value, today_value], y=[real_value, learned_guess], mode="lines",
    line=dict(color="crimson", dash="dot", width=3), name="Gap (dot to line)",
))

# Three big markers on the example day: real, persistence, model
for value, colour in [(real_value, "crimson"), (persistence_guess, "royalblue"), (learned_guess, "darkorange")]:
    gap_figure.add_trace(go.Scatter(
        x=[today_value], y=[value], mode="markers", showlegend=False,
        marker=dict(color=colour, size=13, line=dict(color="black", width=1)),
    ))

# Write the numbers on the chart, next to each marker
gap_figure.add_annotation(x=today_value, y=real_value, xanchor="left", xshift=14, showarrow=False,
                          text=f"<b>real tomorrow = {real_value:.0f}</b>", font=dict(color="crimson"))
gap_figure.add_annotation(x=today_value, y=persistence_guess, xanchor="left", xshift=14, showarrow=False,
                          text=f"persistence says {persistence_guess:.0f} (off by {abs(persistence_guess - real_value):.0f})",
                          font=dict(color="royalblue"))
gap_figure.add_annotation(x=today_value, y=learned_guess, xanchor="right", xshift=-14, showarrow=False,
                          text=f"model says {learned_guess:.0f} (off by {abs(learned_guess - real_value):.0f})",
                          font=dict(color="darkorange"))

# Zoom in: a few extreme days at 400-600 would squash everything else
gap_figure.update_layout(
    title=f"{example_day.date()}: today = {today_value:.0f}. Red dotted = how far each guess is from the real value",
    xaxis_title="Today's PM2.5 until 17:00 (across)", yaxis_title="Tomorrow's real PM2.5 (up)",
    xaxis_range=[40, 160], yaxis_range=[30, 150], legend=dict(orientation="h", y=-0.2),
)
gap_figure.show()


Example day: 2025-06-07
Today until 17:00 = 100   |   real tomorrow = 74
Persistence says 100  -> off by 26
Model says 107  -> off by 33


### 7.4 Predict on the test days and score with MAE

The model has learned its line from the train rows. Now comes the real exam:
the 162 test days from 2026-03-01 onwards, which it has **never seen**.

```text
X_test (today's air) ──► model.predict() ──► model_says
                                              │  compare with
y_test  (real tomorrow) ───────────────────────┘
                                              │
                                    gap = |model_says − real|
                                              │
                                    MAE = average of all gaps
```

`model.predict(X_test)` applies the learned line to every test day:
`0.80 × today + 27`. Nothing is learned here, the two numbers are fixed.

For each day we compute the **gap**, the distance between the prediction and the
real value, with the minus sign dropped. MAE is simply the average of those gaps.
This is the same MAE we calculated for the baselines in Step 5, so the numbers can
be compared directly.

We calculate the MAE two ways, by averaging our `gap` column and with
`mean_absolute_error`, to show that they give the same answer.

In [29]:
# Ask the model to predict tomorrow for every test day (it has never seen these rows)
test_predictions = model.predict(X_test)

# Put the input, the real answer and the prediction side by side so we can look at them
check = pd.DataFrame({
    "today_until_17": X_test["pm25_until_17"],
    "real_tomorrow": y_test,
    "model_says": test_predictions,
})

# The gap for each day: how far the prediction is from the real value (minus sign dropped)
check["gap"] = (check["model_says"] - check["real_tomorrow"]).abs()

print(check.head())

# MAE = the average of the gap column. We calculate it two ways to see they match.
print("\nTest rows:", len(y_test))
print("Average of the gap column:", round(check["gap"].mean(), 2))
print("mean_absolute_error():    ", round(mean_absolute_error(y_test, test_predictions), 2))


                           today_until_17  real_tomorrow  model_says  \
date                                                                   
2026-03-01 00:00:00+05:30       83.647059      99.950000   93.745223   
2026-03-02 00:00:00+05:30       93.750000      95.458333  101.797079   
2026-03-03 00:00:00+05:30      100.666667      76.666667  107.309534   
2026-03-04 00:00:00+05:30       78.944444      68.791667   89.997327   
2026-03-05 00:00:00+05:30       67.888889      87.500000   81.186254   

                                 gap  
date                                  
2026-03-01 00:00:00+05:30   6.204777  
2026-03-02 00:00:00+05:30   6.338746  
2026-03-03 00:00:00+05:30  30.642867  
2026-03-04 00:00:00+05:30  21.205660  
2026-03-05 00:00:00+05:30   6.313746  

Test rows: 162
Average of the gap column: 19.58
mean_absolute_error():     19.58


### 7.5 Who guesses best: the model or the free rules?

### What we are doing

Think of three people who each guess tomorrow's air for the same 161 days.
Later, we see the real answer and check how wrong each person was.

| Person | How they guess | Did they train? |
|---|---|---|
| `persistence` | "Tomorrow will be like today" | No |
| `yesterday_full_day` | "Tomorrow will be like yesterday" | No |
| `linear_regression` | Uses the line we learned: `0.80 × today + 27` | Yes |

We already know how wrong each one was, in one number: the **MAE** (the average
size of the mistake). A **smaller** MAE means a better guesser.

**So this step asks one simple question: did the person who trained beat the two
who did not?**

### Why the same 161 days?

Persistence and yesterday were checked on 161 days. The model made 162 guesses.
If we compare 161 days with 162 days, it is not fair, like two students taking
different exams. So we keep only the 161 days that all three guessed, and compare
them on those.

### What the code does, in three lines

1. Keep the model's guesses for only those 161 days.
2. Work out each person's MAE against the same real answers.
3. Put them in a table, best (smallest MAE) first, and print the winner.

In [36]:
# The baselines were scored on the rows in `scored` (161 days, no missing value).
# The model must be scored on exactly the same days, or the comparison is unfair.
same_days = scored.index
model_on_same_days = check.loc[same_days, "model_says"]
print("model_on_same_days" , model_on_same_days)

print("Days in check (model):", len(check))
print("Days in scored (baselines):", len(scored))
print("Days we compare on:", len(same_days))

# MAE of the three guesses, all measured against the same real answers
comparison = pd.DataFrame({
    "guess": ["persistence", "yesterday_full_day", "linear_regression"],
    "MAE": [
        mean_absolute_error(scored["actual"], scored["persistence"]),
        mean_absolute_error(scored["actual"], scored["yesterday_full_day"]),
        mean_absolute_error(scored["actual"], model_on_same_days),
    ],
})
comparison = comparison.sort_values("MAE").reset_index(drop=True)   # best (lowest) first

print("\nMAE on the same test days (lower is better):")
print(comparison.to_string(index=False, float_format="%.2f"))

best_guess = comparison.loc[0, "guess"]
print("\nBest guess on these days:", best_guess)


model_on_same_days date
2026-03-01 00:00:00+05:30     93.745223
2026-03-02 00:00:00+05:30    101.797079
2026-03-03 00:00:00+05:30    107.309534
2026-03-04 00:00:00+05:30     89.997327
2026-03-05 00:00:00+05:30     81.186254
                                ...    
2026-09-03 00:00:00+05:30     51.437817
2026-09-05 00:00:00+05:30     52.583477
2026-09-06 00:00:00+05:30     56.752303
2026-09-09 00:00:00+05:30     55.594296
2026-09-10 00:00:00+05:30     55.062975
Name: model_says, Length: 161, dtype: float64
Days in check (model): 162
Days in scored (baselines): 161
Days we compare on: 161

MAE on the same test days (lower is better):
             guess   MAE
       persistence 15.73
yesterday_full_day 16.40
 linear_regression 19.67

Best guess on these days: persistence


### 7.6 What if we shuffle the days? (Ravi's way)

### The story

Asha's friend Ravi says: *"Why keep the days in order? Shuffle them all, use 80 %
to train and 20 % to test. That is what every tutorial does."*

We will try it, and see what happens.

### Two ways to split the same days

| | Time split (what we did) | Random split (Ravi's way) |
|---|---|---|
| Train days | everything **before** 1 Mar 2026 | 80 % of all days, picked at random |
| Test days | everything **from** 1 Mar 2026 | the other 20 %, from all months |
| Is the test in the future? | Yes | **No**, test days are mixed in with the train days |

`train_test_split` does the shuffling for us. `random_state=42` just makes the shuffle
repeat the same way each time we run it.

### Why this is a problem for a forecast

In real life, Asha forecasts **tomorrow**. She never knows next week's air.
With a random split, the model may learn from a day in June and then be tested on a
day in March. It has seen the future. The code prints how many test days are older
than the newest train day, and you will see it is **all of them**.

### What the code does

1. Shuffle and split the days 80 / 20.
2. Train a new `LinearRegression` on the train days (same steps as 7.3).
3. Predict the test days and calculate the MAE (same steps as 7.4).
4. Print the random-split MAE next to the time-split MAE.

**Careful:** the two MAEs are measured on different test days, so do not decide
"which is better" yet. We discuss that in the next step.

In [39]:
from sklearn.model_selection import train_test_split

# Shuffle all the days, then keep 80% for learning and 20% for testing
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X, y, test_size=0.2, random_state=42     # random_state=42 fixes the shuffle, so we get the same result each run
)

print("Train days:", len(X_train_random), "| from", X_train_random.index.min().date(), "to", X_train_random.index.max().date())
print("Test days: ", len(X_test_random), "| from", X_test_random.index.min().date(), "to", X_test_random.index.max().date())

# How many test days are OLDER than the newest train day? (the model learned from days after them)
older_test_days = (X_test_random.index < X_train_random.index.max()).sum()
print("Test days older than the newest train day:", older_test_days, "of", len(X_test_random))

# Same steps as before: create a model, learn from the train days, predict the test days
random_model = LinearRegression()
random_model.fit(X_train_random, y_train_random)
random_predictions = random_model.predict(X_test_random)


random_mae = mean_absolute_error(y_test_random, random_predictions)
print("\nLearned line:", round(random_model.coef_[0], 3), "× today +", round(random_model.intercept_, 2))
print("MAE with the random split:", round(random_mae, 2))
print("MAE with the time split:  ", round(check["gap"].mean(), 2), "(from Step 7.4)")

Train days: 385 | from 2025-02-20 to 2026-09-10
Test days:  97 | from 2025-02-19 to 2026-09-03
Test days older than the newest train day: 97 of 97

Learned line: 0.893 × today + 14.68
MAE with the random split: 31.81
MAE with the time split:   19.58 (from Step 7.4)


### 7.7 Are the two test sets the same kind of days?

In Step 7.6 the MAE was **19.58** for the time split and **31.81** for the random
split. It is tempting to say "the time split is better". But before we say that, we
must ask a fair question:

**Were both models tested on the same kind of days?**

An MAE is an average mistake, and it depends on how hard the days are. Think of two
students: one is tested on easy questions and the other on hard ones. Their scores
cannot be compared.

### What we check

For each test set we print:

| Column | Meaning |
|---|---|
| `days` | How many days are in the test set |
| `average tomorrow` | The average real PM2.5 on those days |
| `highest tomorrow` | The dirtiest day in the set |
| `Poor days (>= 91)` | How many days had Poor-or-worse air |
| `% Poor days` | Those Poor days as a percentage of the test set |

If one test set has dirtier days and bigger spikes, the same model will make bigger
mistakes there, even if the model is no worse.

In [41]:
# Look at the two test sets side by side: are they the same kind of days?
test_sets = {
    "Time split test": y_test,
    "Random split test": y_test_random,
}

summary = []
for name, real_values in test_sets.items():
    summary.append({
        "test set": name,
        "days": len(real_values),
        "average tomorrow": real_values.mean(),
        "highest tomorrow": real_values.max(),
        "Poor days (>= 91)": (real_values >= poor_threshold).sum(),
        "% Poor days": (real_values >= poor_threshold).mean() * 100,
    })
    
summary = pd.DataFrame(summary)
print(summary.to_string(index=False, float_format="%.1f"))



         test set  days  average tomorrow  highest tomorrow  Poor days (>= 91)  % Poor days
  Time split test   162              55.7             148.4                 17         10.5
Random split test    97              84.8             397.4                 24         24.7


#### Try a 70 / 30 random split

What if we change the size of the split? Ravi asks: *"Maybe 80 / 20 was unlucky.
Let's keep 70 % to learn and 30 % to test."*

We use exactly the same code as 7.6. The only change is `test_size=0.3` instead of
`0.2`, so the test set grows from 97 days to 145 days.

The code prints the MAE, and also the same "what kind of days are in the test set"
facts from 7.7. This lets us see whether a different split size fixes the problem,
or whether the score still depends on which days land in the test set.

In [42]:
# Same as 7.6, but keep 70% for learning and 30% for testing
X_train_70, X_test_30, y_train_70, y_test_30 = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Train days:", len(X_train_70), "| Test days:", len(X_test_30))

model_70_30 = LinearRegression()
model_70_30.fit(X_train_70, y_train_70)
predictions_30 = model_70_30.predict(X_test_30)

mae_70_30 = mean_absolute_error(y_test_30, predictions_30)
print("\nLearned line:", round(model_70_30.coef_[0], 3), "× today +", round(model_70_30.intercept_, 2))
print("MAE with the random 70/30 split:", round(mae_70_30, 2))

# What kind of days ended up in this test set?
print("\nTest set average tomorrow:", round(y_test_30.mean(), 1))
print("Test set highest tomorrow:", round(y_test_30.max(), 1))
print("Test set Poor days:", (y_test_30 >= poor_threshold).sum(), "of", len(y_test_30))


Train days: 337 | Test days: 145

Learned line: 0.909 × today + 12.8
MAE with the random 70/30 split: 30.24

Test set average tomorrow: 89.2
Test set highest tomorrow: 397.4
Test set Poor days: 41 of 145


### 7.8 Shuffle the days five times

### The question

In 7.6 we shuffled the days once and got one MAE. But a shuffle is a **roll of the
dice**. If we roll again, do we get the same MAE?

A score you can trust should not change just because you shuffled differently.

### What we do

`random_state` is the number that decides how the days are shuffled. Same number, same
shuffle. A different number gives a different shuffle. We try `random_state` 0, 1, 2, 3
and 4, and each time we:

1. Split the days 80 / 20.
2. Train a new `LinearRegression` on the train days.
3. Score it on the test days, and print the MAE.

We also print the average of the test days each time, because from 7.7 we know the
kind of days in the test set can change the score.

At the end we print the time split MAE next to them. The time split has **no
shuffle** (it always cuts at 2026-03-01), so it gives the same number every run.

### What to look for

- Does the random-split MAE stay the same, or jump around?
- Is the time-split MAE stable?

In [44]:
# Shuffle the days 5 different ways (a different random_state = a different shuffle)
random_results = []
for seed in range(5):
    X_a, X_b, y_a, y_b = train_test_split(X, y, test_size=0.2, random_state=seed)

    shuffle_model = LinearRegression()
    shuffle_model.fit(X_a, y_a)
    shuffle_mae = mean_absolute_error(y_b, shuffle_model.predict(X_b))

    random_results.append({
        "shuffle (random_state)": seed,
        "MAE": shuffle_mae,
        "test average tomorrow": y_b.mean(),
    })
    print(f"random_state={seed}: MAE = {shuffle_mae:.2f}   (test days average {y_b.mean():.1f})")

random_results = pd.DataFrame(random_results)

# The time split has no shuffle, so it gives one number
time_split_mae = mean_absolute_error(y_test, model.predict(X_test))

print("\nRandom split MAE: lowest", round(random_results["MAE"].min(), 2),
      "| highest", round(random_results["MAE"].max(), 2),
      "| spread", round(random_results["MAE"].max() - random_results["MAE"].min(), 2))
print("Time split MAE:   ", round(time_split_mae, 2), "(the same every time we run it)")


random_state=0: MAE = 25.80   (test days average 93.1)
random_state=1: MAE = 30.01   (test days average 102.8)
random_state=2: MAE = 27.80   (test days average 96.9)
random_state=3: MAE = 26.48   (test days average 80.7)
random_state=4: MAE = 29.71   (test days average 90.4)

Random split MAE: lowest 25.8 | highest 30.01 | spread 4.21
Time split MAE:    19.58 (the same every time we run it)
